Notebook to extract tortuosity from impedance data with non-intercalating electrolyte. 
From Biologic mpr files 

Run the first cell to import necessary packages and fitting models

In [ ]:
import yadg
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import xarray as xr
import numpy as np
sys.path.append("../src")  # relative path from notebooks to src
from fitting_models import *

file_path = next(Path("../data").glob("*.mpr"))
name = file_path.stem  
print(f"Processed {name}...")
datatree=yadg.extractors.extract("eclab.mpr",file_path) # use yadg module to extract impedance data from mpr file

Visualize the Nyquist plot. Place markers to restrict the frequency range for fitting 

In [ ]:
freq=datatree["freq"].values # extract frequency from the datatree
Re_Z=datatree["Re(Z)"].values  # extract real part of impedance  
Im_Z = datatree["-Im(Z)"].values # extract imaginary part of impedance

freq_1=10e3  # high intensity marker - change to desired value
freq_2=5e-3  # low intensity marker - change to desired value

#show Nyquist plot
plt.scatter(Re_Z, Im_Z, color="lightgrey") 

#show points closest to 10kHz and 1Hz as dark grey
plt.scatter(Re_Z[np.argmin(np.abs(freq-freq_1))], Im_Z[np.argmin(np.abs(freq-freq_1))], color="blue") 
plt.scatter(Re_Z[np.argmin(np.abs(freq-freq_2))], Im_Z[np.argmin(np.abs(freq-freq_2))], color="blue")

#add 10 kHz and 1Hz labels next to points
plt.text(Re_Z[np.argmin(np.abs(freq-freq_1))], Im_Z[np.argmin(np.abs(freq-freq_1))], f"{freq_1} Hz", fontsize=8, verticalalignment='bottom', horizontalalignment='right')
plt.text(Re_Z[np.argmin(np.abs(freq-freq_2))], Im_Z[np.argmin(np.abs(freq-freq_2))], f"{freq_2} Hz", fontsize=8, verticalalignment='bottom', horizontalalignment='right')

plt.axis('equal')
plt.show()

Define the constants of your cell (!), the initial guess of your fit, the fit boundary limits, and run this cell for fitting your data to the equivalent circuit model

In [ ]:
# Define constants
L= 0.0119 #cm thickness of one electrode, without current collector
A= 0.95 #cm2 apparent area of electrode
k0=0.0429E-3 #S/cm #bulk conductivity of electrolyte
por=0.27 #porosity of electrode

# --- Create mask between freq_2 and freq_1 ---
mask = (freq <= freq_1) & (freq >= freq_2)

# Apply mask
freq_masked = freq[mask]
Re_Z_masked = Re_Z[mask]
Im_Z_masked = -Im_Z[mask]


# Angular frequency conversion
w_data = 2 * np.pi * np.array(freq_masked)

# Prepare the data for fitting
y_data = np.hstack((Re_Z_masked, Im_Z_masked))  # concatenated real and imaginary parts

# Fitting process
num_runs = 1000  # Set the number of runs
params = None  # Initialize params 

# Initial guess for fitting parameters
#        (ri [ohm cm], qct [F cm-3], nct, Rcc [ohm],Qcc [F], ncc, R0 [ohm])
first_guess=(120000,1e-2,0.9,50,1e-3,0.9,300)

for _ in range(num_runs):
    # Use the previous params as the initial guess for the next runfirst_guess=(1e3,1,1,1e-5,0.9,1e-5,0.9,10)
    initial_guess = params if params is not None else (first_guess)
    
    # Fit the data with increased maxfev
    params, covariance = curve_fit(
        lambda w, ri,qct,nct,Rcc,Qcc,ncc,R0: R0_TLMZ_cc_fit(L,A,w, ri,qct,nct,Rcc,Qcc,ncc,R0), #L,A fixed in fitting function - already defined
        w_data, # frequences
        y_data, # experimental data to fit - concatenated real and imaginary parts
        p0=initial_guess, 
        bounds=((0,1e-8,0,0,1e-8,0,0),(1000000,1,1,10000,1,1,10000)), # set bounds for parameters - same order as in initial_guess
        maxfev=100000 
        )

    # Extract fitted parameters
ri_fit, qct_fit, nct_fit, Rcc_fit, Qcc_fit, ncc_fit, R0_fit = params
# tort=k0*ri_fit*por  #ri_fit is effective resistivity of electrolyte in pore network already expressed in Ohm cm
keff=1/ri_fit #[S cm-1] convert apparent resistivity ri to apparent conductivity keff
tort=k0/keff*por  
Qct=qct_fit*A*L  #convert qct from per volume to total capacitance


# Print fitting parameters
print("Electrode tortuosity factor:")
print(f"{tort}")
print("Current collector contact resistance:")
print(f"Rcc:{Rcc_fit} Ohm")
print("Other fitted constants:")
print(f"Qct:{Qct} F, Qcc:{Qcc_fit} F, ncc:{ncc_fit}, R0:{R0_fit} Ohm")
print(f"ri:{ri_fit}, qct:{qct_fit}, nct:{nct_fit}, Rcc:{Rcc_fit}, Qcc:{Qcc_fit}, ncc:{ncc_fit}, R0:{R0_fit}")

# Generate fitted curve for plotting
EIS_fitting_plot = R0_fit+2*TLMZ_cc(L,A,w_data, *params[0:-1])
EIS_start_plot = first_guess[-1]+2*TLMZ_cc(L,A,w_data, *first_guess[0:-1])

# Plot the results
plt.scatter(Re_Z_masked, Im_Z_masked, label='Measured Data', color='grey')
plt.plot(np.real(EIS_fitting_plot), np.imag(EIS_fitting_plot), label='Fitted Function', color='red')
plt.plot(np.real(EIS_start_plot), np.imag(EIS_start_plot), label='Start Function', color='green')
plt.xlabel("Z' ($\Omega$)")
plt.ylabel("Z'' ($\Omega$)")
plt.legend()
ax = plt.gca()
# plt.xlim(0, 2500)   #change limits as needed
# plt.ylim(0, -5000)
ax.set_aspect('equal', adjustable='box')
plt.grid()
plt.show()
